In [0]:
from pyspark.sql import functions as F

SOURCE_TABLE = "retail_demo.silver.fact_sales"

spark.sql("""
CREATE SCHEMA IF NOT EXISTS retail_demo.gold
""")

df = spark.table(SOURCE_TABLE)

print("Silver records:", df.count())

df.printSchema()


In [0]:
gold_sales_kpi = (
    df
    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("total_customers"),

        F.sum("quantity")
        .alias("total_quantity"),

        F.round(
            F.sum("gross_sales"), 2
        ).alias("gross_sales"),

        F.round(
            F.sum("discount_amount"), 2
        ).alias("discount_amount"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("net_sales"),

        F.round(
            F.sum("tax_amount"), 2
        ).alias("tax_amount"),

        F.round(
            F.sum("order_value"), 2
        ).alias("total_order_value"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit"),

        F.round(
            F.avg("delivery_days"), 2
        ).alias("avg_delivery_days")
    )

    .withColumn(
        "avg_order_value",
        F.round(
            F.col("net_sales") /
            F.col("total_orders"),
            2
        )
    )

    .withColumn(
        "profit_margin_pct",
        F.round(
            (
                F.col("gross_profit") /
                F.col("net_sales")
            ) * 100,
            2
        )
    )

    .withColumn(
        "_gold_load_timestamp",
        F.current_timestamp()
    )
)

(
    gold_sales_kpi.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.sales_kpi"
    )
)

In [0]:
gold_daily_sales = (
    df

    .groupBy(
        "order_date"
    )

    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers"),

        F.sum("quantity")
        .alias("units_sold"),

        F.round(
            F.sum("gross_sales"), 2
        ).alias("gross_sales"),

        F.round(
            F.sum("discount_amount"), 2
        ).alias("discount_amount"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit")
    )

    .withColumn(
        "avg_order_value",
        F.round(
            F.col("net_sales") /
            F.col("total_orders"),
            2
        )
    )

    .withColumn(
        "profit_margin_pct",
        F.round(
            (
                F.col("gross_profit") /
                F.col("net_sales")
            ) * 100,
            2
        )
    )
)

(
    gold_daily_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.daily_sales"
    )
)

In [0]:
gold_monthly_sales = (
    df

    .groupBy(
        "order_year",
        "order_month",
        "order_month_name"
    )

    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers"),

        F.sum("quantity")
        .alias("units_sold"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit"),

        F.round(
            F.sum("discount_amount"), 2
        ).alias("discount_amount")
    )

    .withColumn(
        "avg_order_value",
        F.round(
            F.col("net_sales") /
            F.col("total_orders"),
            2
        )
    )

    .withColumn(
        "profit_margin_pct",
        F.round(
            (
                F.col("gross_profit") /
                F.col("net_sales")
            ) * 100,
            2
        )
    )
)

(
    gold_monthly_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.monthly_sales"
    )
)

In [0]:
gold_product_performance = (
    df

    .groupBy(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand"
    )

    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.sum("quantity")
        .alias("units_sold"),

        F.countDistinct("customer_id")
        .alias("unique_customers"),

        F.round(
            F.sum("gross_sales"), 2
        ).alias("gross_sales"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit"),

        F.round(
            F.avg("discount_pct") * 100,
            2
        ).alias("avg_discount_pct")
    )

    .withColumn(
        "profit_margin_pct",
        F.round(
            (
                F.col("gross_profit") /
                F.col("net_sales")
            ) * 100,
            2
        )
    )
)

(
    gold_product_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.product_performance"
    )
)

In [0]:
gold_customer_performance = (
    df

    .groupBy(
        "customer_id",
        "customer_name",
        "segment",
        "loyalty_member",
        "city",
        "state",
        "region"
    )

    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.sum("quantity")
        .alias("units_purchased"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("total_spend"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit"),

        F.round(
            F.avg("net_sales"), 2
        ).alias("avg_transaction_value"),

        F.min("order_date")
        .alias("first_order_date"),

        F.max("order_date")
        .alias("last_order_date")
    )
)

gold_customer_performance = (
    gold_customer_performance

    .withColumn(
        "customer_value_segment",

        F.when(
            F.col("total_spend") >= 100000,
            "High Value"
        )

        .when(
            F.col("total_spend") >= 50000,
            "Medium Value"
        )

        .otherwise(
            "Standard"
        )
    )
)
(
    gold_customer_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.customer_performance"
    )
)

In [0]:
gold_regional_performance = (
    df

    .groupBy(
        "region",
        "state",
        "city"
    )

    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers"),

        F.sum("quantity")
        .alias("units_sold"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit"),

        F.round(
            F.avg("delivery_days"), 2
        ).alias("avg_delivery_days")
    )

    .withColumn(
        "avg_order_value",
        F.round(
            F.col("net_sales") /
            F.col("total_orders"),
            2
        )
    )
)

(
    gold_regional_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.regional_performance"
    )
)

In [0]:
gold_channel_performance = (
    df

    .groupBy(
        "sales_channel"
    )

    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers"),

        F.sum("quantity")
        .alias("units_sold"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit"),

        F.round(
            F.avg("delivery_days"), 2
        ).alias("avg_delivery_days")
    )

    .withColumn(
        "avg_order_value",
        F.round(
            F.col("net_sales") /
            F.col("total_orders"),
            2
        )
    )
)

(
    gold_channel_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.channel_performance"
    )
)

In [0]:
gold_campaign_performance = (
    df

    .filter(
        F.col("campaign_id").isNotNull()
    )

    .groupBy(
        "campaign_id",
        "campaign_name",
        "marketing_channel",
        "objective",
        "budget_inr"
    )

    .agg(

        F.countDistinct("order_id")
        .alias("total_orders"),

        F.countDistinct("customer_id")
        .alias("unique_customers"),

        F.round(
            F.sum("net_sales"), 2
        ).alias("net_sales"),

        F.round(
            F.sum("gross_profit"), 2
        ).alias("gross_profit")
    )

    .withColumn(
        "sales_to_budget_ratio",
        F.round(
            F.col("net_sales") /
            F.col("budget_inr"),
            2
        )
    )
)

(
    gold_campaign_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "retail_demo.gold.campaign_performance"
    )
)

In [0]:
gold_tables = [
    "sales_kpi",
    "daily_sales",
    "monthly_sales",
    "product_performance",
    "customer_performance",
    "regional_performance",
    "channel_performance",
    "campaign_performance"
]

for table in gold_tables:

    count = spark.table(
        f"retail_demo.gold.{table}"
    ).count()

    print(
        f"{table}: {count:,} records"
    )